In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
print(df.head())
print(df.info())

    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0   

In [2]:
print(df.isnull().sum())

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


In [3]:
df = df.drop(columns = ['Loan_ID'])

In [4]:
df['Gender'] = df['Gender'].map({"Male": 0, "Female": 1})
df['Married'] = df['Married'].map({"Yes": 1, "No": 0})
df['Loan_Status'] = df['Loan_Status'].map({"Y": 1, "N": 0})
df['Education'] = df['Education'].map({"Graduate": 1, "Not Graduate": 0})
df['Self_Employed'] = df['Self_Employed'].map({'Yes':1, "No": 0})


In [5]:
binary_features = ["Gender", "Married", "Education", "Self_Employed", "Credit_History"]
categorical_features = ["Dependents", "Property_Area"]
numeric_features = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]

In [6]:
print(df['Loan_Status'].value_counts())

Loan_Status
1    422
0    192
Name: count, dtype: int64


IMBALANCED

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42, stratify=  y)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop = 'first'))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [9]:
from sklearn.compose import ColumnTransformer
preprocesor =  ColumnTransformer([
    ("bin", binary_pipeline, binary_features), 
    ("cat", categorical_pipeline, categorical_features),
    ("num", numeric_pipeline, numeric_features)
])

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": LogisticRegression(max_iter= 1000, random_state=42, class_weight = 'balanced'), 
    "Decision Trees": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight= 'balanced')
}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocesor', preprocesor),
        ('model', model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv = 5, scoring = 'f1')
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

Logistic Regression: 0.8056 +/- 0.0147
Decision Trees: 0.7820 +/- 0.0272
Random Forest: 0.8451 +/- 0.0127


In [11]:
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocesor', preprocesor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc = accuracy_score(y_test, pipe.predict(X_test))
    print(f"{name}: Test: {test_acc} | Train: {train_acc}")

Logistic Regression: Test: 0.8292682926829268 | Train: 0.7576374745417516
Decision Trees: Test: 0.7723577235772358 | Train: 1.0
Random Forest: Test: 0.8292682926829268 | Train: 1.0


In [12]:
lr_pipeline = fitted_pipelines["Logistic Regression"]
y_pred = lr_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.73      0.71      0.72        38
           1       0.87      0.88      0.88        85

    accuracy                           0.83       123
   macro avg       0.80      0.80      0.80       123
weighted avg       0.83      0.83      0.83       123

[[27 11]
 [10 75]]


s